<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/05_Hands_On_Labs/notebooks/09_neural_network_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Neural Networks from Scratch with NumPy

> **Track:** Data Science / ML Engineer | **Level:** Intermediate | **Time:** 2-3 hours

## Overview
Understanding backpropagation at the math level makes you a better ML practitioner — you'll debug gradient issues, understand optimizers, and never be mystified by what a framework does under the hood. We implement everything from scratch using only NumPy.

### What You'll Learn
- Activation functions (sigmoid, ReLU, tanh)
- Forward pass implementation
- Cross-entropy loss and mean squared error
- Backpropagation (chain rule step by step)
- Gradient descent and mini-batch SGD
- Training on a real dataset
- Comparison with sklearn MLP

```bash
pip install numpy matplotlib scikit-learn
```

In [ ]:
# Setup: imports and activation functions
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional

np.random.seed(42)

# Core activation functions and their derivatives
def sigmoid(z: np.ndarray) -> np.ndarray:
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def sigmoid_deriv(a: np.ndarray) -> np.ndarray:
    """Derivative given ACTIVATED value a = sigmoid(z)."""
    return a * (1 - a)

def relu(z: np.ndarray) -> np.ndarray:
    return np.maximum(0, z)

def relu_deriv(z: np.ndarray) -> np.ndarray:
    return (z > 0).astype(float)

def softmax(z: np.ndarray) -> np.ndarray:
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

# Visualize activations
x = np.linspace(-5, 5, 200)
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, (fn, name) in zip(axes, [(sigmoid, "Sigmoid"), (relu, "ReLU"), (np.tanh, "Tanh")]):
    y = fn(x)
    ax.plot(x, y, 'b-', linewidth=2)
    ax.axhline(0, color='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.5)
    ax.set_title(name)
    ax.set_ylim(-1.5, 1.5)
plt.tight_layout()
plt.show()

## 1. Forward Pass

The forward pass computes the prediction: `input → hidden layers → output`. We cache intermediate values for backprop.

In [ ]:
# Two-layer neural network: forward pass with cache

def initialize_params(layer_dims: list[int]) -> dict[str, np.ndarray]:
    """Xavier/He initialization for a network with given layer dimensions."""
    params: dict[str, np.ndarray] = {}
    for l in range(1, len(layer_dims)):
        scale = np.sqrt(2.0 / layer_dims[l-1])  # He initialization
        params[f"W{l}"] = np.random.randn(layer_dims[l-1], layer_dims[l]) * scale
        params[f"b{l}"] = np.zeros((1, layer_dims[l]))
    return params

def forward_pass(
    X: np.ndarray,
    params: dict[str, np.ndarray],
    n_layers: int
) -> tuple[np.ndarray, dict[str, np.ndarray]]:
    """Forward pass: returns output and cache of intermediate values."""
    cache: dict[str, np.ndarray] = {"A0": X}
    A = X
    for l in range(1, n_layers):
        Z = A @ params[f"W{l}"] + params[f"b{l}"]
        A = relu(Z) if l < n_layers - 1 else sigmoid(Z)
        cache[f"Z{l}"] = Z
        cache[f"A{l}"] = A
    return A, cache

def compute_loss(y_pred: np.ndarray, y_true: np.ndarray) -> float:
    """Binary cross-entropy loss."""
    eps = 1e-8
    return -np.mean(y_true * np.log(y_pred + eps) + (1 - y_true) * np.log(1 - y_pred + eps))

# Test forward pass
layer_dims = [4, 8, 4, 1]  # 4 inputs, 2 hidden layers, 1 output
params = initialize_params(layer_dims)
X_test = np.random.randn(5, 4)
y_test = np.array([[1], [0], [1], [0], [1]])
y_pred, cache = forward_pass(X_test, params, len(layer_dims))
loss = compute_loss(y_pred, y_test)
print(f"Forward pass output shape: {y_pred.shape}")
print(f"Initial loss (random weights): {loss:.4f}")

## 2. Backpropagation

Backprop applies the **chain rule** to compute gradients of the loss with respect to every parameter, propagating error backwards through the network.

In [ ]:
# Backpropagation for a 2-hidden-layer network

def backward_pass(
    y_true: np.ndarray,
    params: dict[str, np.ndarray],
    cache: dict[str, np.ndarray],
    n_layers: int
) -> dict[str, np.ndarray]:
    """Compute gradients via backpropagation."""
    m = y_true.shape[0]
    grads: dict[str, np.ndarray] = {}

    # Output layer gradient (cross-entropy + sigmoid)
    L = n_layers - 1
    dA = cache[f"A{L}"] - y_true  # combined CE + sigmoid derivative

    for l in range(L, 0, -1):
        A_prev = cache[f"A{l-1}"]
        grads[f"dW{l}"] = A_prev.T @ dA / m
        grads[f"db{l}"] = np.mean(dA, axis=0, keepdims=True)

        if l > 1:  # propagate to previous layer
            dZ_prev = dA @ params[f"W{l}"].T
            dA = dZ_prev * relu_deriv(cache[f"Z{l-1}"])  # ReLU derivative for hidden

    return grads

def update_params(
    params: dict[str, np.ndarray],
    grads: dict[str, np.ndarray],
    lr: float
) -> dict[str, np.ndarray]:
    """Gradient descent parameter update."""
    updated = {k: v.copy() for k, v in params.items()}
    for key in params:
        updated[key] = params[key] - lr * grads[f"d{key}"]
    return updated

# Test backward pass
grads = backward_pass(y_test, params, len(layer_dims))
print("Gradient shapes:")
for k, v in grads.items():
    print(f"  {k}: {v.shape}")

## 3. Training on Real Data

In [ ]:
# Full training loop: train on sklearn's breast cancer dataset
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler as SKScaler

# Load and preprocess data
data = load_breast_cancer()
X_all = data.data.astype(float)
y_all = data.target.reshape(-1, 1).astype(float)

X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.2, random_state=42)
scaler = SKScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Initialize network: 30 features → 16 → 8 → 1
dims = [30, 16, 8, 1]
params = initialize_params(dims)
n_layers = len(dims)

# Training loop with mini-batch SGD
lr = 0.01
epochs = 200
batch_size = 32
losses: list[float] = []

for epoch in range(epochs):
    idx = np.random.permutation(len(X_train))
    for i in range(0, len(X_train), batch_size):
        Xb = X_train[idx[i:i+batch_size]]
        yb = y_train[idx[i:i+batch_size]]
        y_pred, cache = forward_pass(Xb, params, n_layers)
        grads = backward_pass(yb, params, cache, n_layers)
        params = update_params(params, grads, lr)

    if epoch % 20 == 0:
        full_pred, _ = forward_pass(X_train, params, n_layers)
        loss = compute_loss(full_pred, y_train)
        losses.append(loss)
        print(f"Epoch {epoch:3d} | Loss: {loss:.4f}")

# Evaluate
y_pred_test, _ = forward_pass(X_test, params, n_layers)
accuracy = np.mean((y_pred_test > 0.5).astype(int) == y_test)
print(f"\nTest Accuracy: {accuracy:.4f}")

## 4. Comparison with sklearn MLP

In [ ]:
# Compare scratch implementation vs sklearn MLPClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# Our scratch model
scratch_preds = (y_pred_test > 0.5).astype(int).flatten()
scratch_probs = y_pred_test.flatten()

# sklearn MLP with same architecture
mlp = MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=200, random_state=42)
mlp.fit(X_train, y_train.ravel())
sklearn_preds = mlp.predict(X_test)
sklearn_probs = mlp.predict_proba(X_test)[:, 1]

print("=== Model Comparison ===")
print(f"{'Model':<20} {'Accuracy':>10} {'AUC-ROC':>10}")
print("-" * 42)
for name, preds, probs in [
    ("NumPy (scratch)", scratch_preds, scratch_probs),
    ("sklearn MLP", sklearn_preds, sklearn_probs)
]:
    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)
    print(f"{name:<20} {acc:>10.4f} {auc:>10.4f}")

## Exercises

1. **Add momentum**: Implement SGD with momentum by keeping a velocity vector for each parameter: `v = β*v - lr*grad` and `param = param + v`. Use β=0.9. Compare convergence speed vs. vanilla SGD on the breast cancer dataset.

2. **Implement Adam optimizer**: Add Adam to your training loop (track first and second moment estimates for each parameter). Verify that it converges faster than SGD. Plot the training loss curve for both optimizers side by side.

3. **Multi-class classification**: Modify the network to handle multi-class output using softmax activation and categorical cross-entropy loss. Train it on the iris dataset (3 classes, 4 features). Implement one-hot encoding for the labels and verify the gradient derivation for the softmax + CE combination.